In [ ]:
from datetime import datetime
from getpass import getpass

rdm_url = 'http://localhost:5001/'
idp_name_1 = 'FakeCAS'
idp_username_1 = None
idp_password_1 = None
project_name = 'TEST-WORKFLOW-202512251532'
engine_name = 'TEST-ENGINE-202512251538-Updated'
workflow_zip_path = 'resources/simple-approval-workflow.zip'
workflow_name = None
default_result_path = None
close_on_fail = False
transition_timeout = 30000
browser_type = 'chromium'
ignore_https_errors = False

In [ ]:
import tempfile

if idp_username_1 is None:
    idp_username_1 = input(prompt=f'Username for {idp_name_1}')
if idp_password_1 is None:
    idp_password_1 = getpass(prompt=f'Password for {idp_username_1}@{idp_name_1}')

if project_name is None:
    project_name = input(prompt='Project name')
if engine_name is None:
    engine_name = input(prompt='Workflow engine name')
if workflow_zip_path is None:
    workflow_zip_path = input(prompt='Workflow ZIP path')
if workflow_name is None:
    workflow_name = datetime.now().strftime('TEST-TEMPLATE-%Y%m%d%H%M')

# ワークフローテンプレートの登録

- サブシステム名: アドオン
- ページ/アドオン: RDMワークフロー
- 機能分類: ワークフローテンプレート管理
- シナリオ名: テンプレートの登録
- 用意するテストデータ: アカウント、プロジェクト名、エンジン名、ワークフローZIPファイル
- 事前条件: エンジン登録が完了しており、対象プロジェクトがアップロード許可ノードIDに登録されていること

In [ ]:
work_dir = tempfile.mkdtemp()
if default_result_path is None:
    default_result_path = work_dir
work_dir

In [ ]:
import importlib
import pandas as pd

import scripts.playwright
importlib.reload(scripts.playwright)

from scripts.playwright import *
from scripts import grdm

await init_pw_context(close_on_fail=close_on_fail, last_path=default_result_path, ignore_https_errors=ignore_https_errors)

## 新規タブを開き、GRDMトップページを表示する

GRDMトップページが表示されること

In [ ]:
async def _step(page):
    await page.goto(rdm_url)
    await page.locator('//button[text() = "同意する"]').click()
    await expect(page.locator('//button[text() = "同意する"]')).to_have_count(0, timeout=500)

await run_pw(_step, new_page=True)

## ログイン情報を用いてGakuNin RDMにログインする

GRDMダッシュボードが表示されること

In [ ]:
async def _step(page):
    await grdm.login(page, idp_name_1, idp_username_1, idp_password_1, transition_timeout=transition_timeout)
    await grdm.expect_dashboard(page, transition_timeout=transition_timeout)

await run_pw(_step)

## ダッシュボードのプロジェクト一覧から対象プロジェクトをクリックする

プロジェクトダッシュボードが表示されること

In [ ]:
async def _step(page):
    await page.locator(f'//*[@data-test-dashboard-item-title and text() = "{project_name}"]').click()
    await expect(page.locator('//span[@id = "nodeTitleEditable"]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## プロジェクトダッシュボードの上部メニューから「アドオン」をクリックする

アドオン設定画面が表示されること

In [ ]:
async def _step(page):
    await page.locator('//a[text() = "アドオン"]').click()
    await expect(page.locator('//h3[text() = "アドオンを選択"]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 「アドオンを選択」のパネル内「Workflow」の行の「有効にする」をクリックする

「Workflowアドオン規約」のダイアログが表示されること

In [ ]:
async def _step(page):
    enable_button = page.locator('//div[@full_name = "Workflow"]//a[text() = "有効にする"]')
    if await enable_button.count():
        await enable_button.click()
        await expect(page.locator('//button[@data-bb-handler = "confirm"]')).to_be_visible(timeout=transition_timeout)
    else:
        print('Workflow addon is already enabled for this project')

await run_pw(_step)

## 「確認」をクリックする

「アドオンを構成」のパネル内に「Workflow」の行が追加されること

In [ ]:
async def _step(page):
    confirm_button = page.locator('//button[@data-bb-handler = "confirm"]')
    if await confirm_button.count():
        await confirm_button.click()
    await expect(page.locator('#workflow-engine-id')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 「ワークフローエンジン」ドロップダウンからエンジンを選択する

指定したエンジンが選択されること

In [ ]:
async def _step(page):
    option = page.locator(f'#workflow-engine-id option:has-text("{engine_name}")')
    value = await option.get_attribute('value')
    await page.locator('#workflow-engine-id').select_option(value=value)

await run_pw(_step)

## 「ワークフローZIP」にBPMNファイルを含むZIPをアップロードする

ZIPファイルが選択されること

In [ ]:
async def _step(page):
    await page.locator('#workflow-zip').set_input_files(workflow_zip_path)

await run_pw(_step)

## 「名前」にワークフロー名を入力する

ワークフロー名が入力されること

In [ ]:
async def _step(page):
    await page.locator('#workflow-label').fill(workflow_name)

await run_pw(_step)

## 「管理者トークン」ドロップダウンで「閲覧＆編集権限で使用」を選択する

「閲覧＆編集権限で使用」が選択されること

In [ ]:
async def _step(page):
    await page.locator('select[data-bind="value: form.managerTokenMode"]').select_option(value='readwrite')

await run_pw(_step)

## 「ワークフローテンプレートを登録」をクリックする

テンプレートが作成され、有効化用ドロップダウンに表示されること

In [ ]:
async def _step(page):
    await page.locator('//button[.//span[contains(text(), "ワークフローテンプレートを登録")]]').click()
    await expect(page.locator(f'#activate-workflow-select option:has-text("{workflow_name}")')).to_be_attached(timeout=transition_timeout)

await run_pw(_step)

## 後始末

In [ ]:
await finish_pw_context(screenshot=False, last_path=default_result_path)

In [ ]:
!rm -fr {work_dir}